In [1]:
import pandas as pd

# Path points up one directory, then into data/
df = pd.read_csv("cob_dataset.csv")

print("Dataset Columns:", df.columns.tolist())
df.head()

Dataset Columns: ['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Responsibilities', 'Keywords']


,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...
3,NET-F-004,.NET Developer,Fresher,0-1,C#; .NET Framework; ASP.NET basics; SQL Server...,Support in software design documentation; Assi...,.NET; C#; SQL Server; Entity Framework; ASP.NET
4,NET-F-005,.NET Developer,Fresher,0-1,C#; ASP.NET; MVC; Entity Framework basics; SQL...,Learn to design and build ASP.NET applications...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


In [4]:
# Keep the essential columns
df_clean = df[['Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Keywords']].copy()

In [ ]:
# Fill any missing values with empty strings so concatenation doesn't break
df_clean['Skills'] = df_clean['Skills'].fillna('')
df_clean['Keywords'] = df_clean['Keywords'].fillna('')

In [9]:
# Combine Skills and Keywords into a single master string
df_clean['Combined_Skills'] = df_clean['Skills'] + ";" + df_clean['Keywords']

In [10]:
# Create a function to clean, lowercase, and deduplicate the skills
def clean_and_deduplicate(skill_str):
    # Replace commas with semicolons just in case, then split by semicolon
    skill_str = skill_str.replace(',', ';')
    
    # Lowercase, strip whitespace, and filter out empty strings
    skills_list = [s.strip().lower() for s in skill_str.split(';') if s.strip()]
    
    # Remove duplicates by converting to a dictionary (preserves order) and back to list
    unique_skills = list(dict.fromkeys(skills_list))
    
    # Join back into a clean, comma-separated string
    return ', '.join(unique_skills)

In [11]:
# Apply the cleaning function
df_clean['Target_Skills'] = df_clean['Combined_Skills'].apply(clean_and_deduplicate)

# Standardize the formatting for Title and Experience Level
df_clean['Title'] = df_clean['Title'].str.strip().str.title()
df_clean['ExperienceLevel'] = df_clean['ExperienceLevel'].str.strip().str.title()
df_clean['YearsOfExperience'] = df_clean['YearsOfExperience'].str.strip()

In [12]:
# Keep only the final, cleaned columns
df_final = df_clean[['Title', 'ExperienceLevel', 'YearsOfExperience', 'Target_Skills']]

# Drop exact duplicate rows (e.g., if there are 5 identical "Fresher .NET Developer" rows)
df_final = df_final.drop_duplicates()

# Export the clean data for Django and Hugging Face to use!
df_final.to_csv("../data/cleaned_tech_skills.csv", index=False)

print(f"Success! Cleaned dataset saved with {len(df_final)} unique career profiles.")
df_final.head(10)

Success! Cleaned dataset saved with 1008 unique career profiles.


,Title,ExperienceLevel,YearsOfExperience,Target_Skills
0,.Net Developer,Fresher,0-1,"c#, vb.net basics, .net framework, .net core f..."
1,.Net Developer,Fresher,0-1,"c#, .net framework basics, asp.net, razor, htm..."
2,.Net Developer,Fresher,0-1,"c#, vb.net basics, .net core, asp.net mvc, htm..."
3,.Net Developer,Fresher,0-1,"c#, .net framework, asp.net basics, sql server..."
4,.Net Developer,Fresher,0-1,"c#, asp.net, mvc, entity framework basics, sql..."
5,.Net Developer,Fresher,0-1,"c#, .net framework, asp.net, razor, sql server..."
6,.Net Developer,Fresher,0-1,"c#, vb.net, .net core basics, asp.net mvc, htm..."
7,.Net Developer,Fresher,0-1,"c#, .net framework basics, asp.net mvc, entity..."
8,.Net Developer,Fresher,0-1,"c#, .net framework, asp.net, html, css, sql se..."
9,.Net Developer,Fresher,0-1,"c#, asp.net mvc, .net core basics, entity fram..."


In [18]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv("../data/job_dataset.csv")

# 1. Keep the essential columns
df_clean = df[[ 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Responsibilities', 'Keywords']].dropna().copy()

# 2. CLEAN THE TITLE COLUMN
df_clean['Title'] = df_clean['Title'].apply(lambda x: str(x).split(' - ')[0].strip())

# Remove seniority prefixes/suffixes to make base titles perfectly uniform
prefixes_to_remove = ['Junior ', 'Senior ', 'Lead ', 'Principal ', 'Entry-level ', 'Associate ', ' Intern', ' Trainee']
for word in prefixes_to_remove:
    df_clean['Title'] = df_clean['Title'].str.replace(word, '', case=False).str.strip()

# Standardize the Skills and Keywords (lowercase, replace ';' with ',')
def format_text(text_str):
    if not isinstance(text_str, str):
        return ""
    return text_str.lower().replace(';', ',')

df_clean['Skills'] = df_clean['Skills'].apply(format_text)
df_clean['Keywords'] = df_clean['Keywords'].apply(format_text)

# Create the final "Target_Skills" column for the AI
df_clean['Target_Skills'] = df_clean['Skills'] + ", " + df_clean['Keywords']
df_clean['Target_Skills'] = df_clean['Target_Skills'].str.replace(r'\s*,\s*', ', ', regex=True).str.strip(', ')


In [19]:
# Export the clean data
df_clean.to_csv("../data/cleaned_tech_skills2.csv", index=False)

print(f"Success! Exported {len(df_clean)} rows with perfectly clean titles.")
print(df_clean['Title'].unique()[:10])

Success! Exported 1067 rows with perfectly clean titles.
<StringArray>
[               '.NET Developer',                   'AI Engineer',
            'AI Prompt Engineer',             'Android Developer',
         'Android App Developer', 'Trainee Android App Developer',
              'Android Engineer',           'Android Development',
    'Android Solutions Engineer',             'Android Tech Lead']
Length: 10, dtype: str
